# Stochastic Processes in Quantitative Finance
**Module 01 — Abubakar Mamudu Alutiba**  
MSc Financial Engineering, WorldQuant University

---

Stochastic processes are mathematical models for systems that evolve randomly over time.  
In finance, they are the foundation of everything from stock price models to interest rate theory.

This notebook walks through three foundational processes — **from first principles** — with simulation, visualisation, and validation against theory.

| Process | SDE | Key use in finance |
|---|---|---|
| Brownian Motion | $dW = \sqrt{dt} \cdot Z$ | Foundation of all continuous-time models |
| Geometric Brownian Motion | $dS = \mu S\,dt + \sigma S\,dW$ | Stock prices — Black-Scholes model |
| Ornstein-Uhlenbeck | $dX = \theta(\mu - X)dt + \sigma\,dW$ | Interest rates, commodity prices, pairs trading |

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import norm, lognorm

from src.models import BrownianMotion, GeometricBrownianMotion, OrnsteinUhlenbeck
from src.visualization import plot_brownian_motion, plot_gbm, plot_ou

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
print('All imports successful ✅')

---
## 1. Brownian Motion (Wiener Process)

Brownian Motion is the building block of all continuous-time stochastic models in finance.  
It was originally used to describe the random movement of pollen particles in water — observed by Robert Brown in 1827 and formalised mathematically by Norbert Wiener in 1923.

### Definition
A standard Brownian Motion $W(t)$ satisfies:
- $W(0) = 0$ — starts at zero
- $W(t) - W(s) \sim N(0, t-s)$ for $s < t$ — increments are normally distributed
- Independent increments — what happened before doesn't affect what happens next
- Continuous paths (almost surely)

### Key theoretical properties
$$E[W(t)] = 0 \qquad \text{Var}[W(t)] = t \qquad \text{std}[W(t)] = \sqrt{t}$$

The variance grows **linearly** with time — paths spread out as $\sqrt{t}$, not linearly.

In [ ]:
# Simulate 500 paths of Brownian Motion over 1 year (252 trading days)
bm = BrownianMotion(T=1.0, N=252, n_paths=500, seed=42)
W  = bm.simulate()

print(f"Simulation output shape : {W.shape}  →  (n_paths, N+1)")
print(f"All paths start at zero : {np.all(W[:, 0] == 0)}")
print()
print(f"E[W(T)] simulated  : {W[:, -1].mean():.4f}   (theory = 0)")
print(f"std[W(T)] simulated: {W[:, -1].std():.4f}   (theory = {np.sqrt(bm.T):.4f})")

Notice how the simulated mean is extremely close to **0** and the simulated std is extremely close to **1.0** (= √T = √1).  
This confirms our simulation correctly implements the theoretical properties of Brownian Motion.

In [ ]:
fig = plot_brownian_motion(bm, W)
plt.show()

**Reading the plots:**
- **Left** — The paths spread out over time, bounded by the ±1σ and ±2σ envelopes which widen as $\sqrt{t}$. The coral path is one random sample.
- **Right** — The terminal distribution W(T=1) matches the theoretical N(0, 1) density perfectly — confirming the simulation is mathematically correct.

---
## 2. Geometric Brownian Motion (GBM)

Brownian Motion can go negative — which makes it unsuitable for modelling stock prices.  
**Geometric Brownian Motion** solves this by modelling the *log* of the price, ensuring $S(t) > 0$ always.

### The SDE
$$dS = \mu S\,dt + \sigma S\,dW$$

Where $\mu$ is the expected return and $\sigma$ is the volatility.

### Exact solution via Itô's Lemma
Applying Itô's Lemma to $f(S) = \log S$ gives the exact solution:
$$S(t) = S_0 \cdot \exp\left(\left(\mu - \frac{\sigma^2}{2}\right)t + \sigma W(t)\right)$$

The $-\frac{\sigma^2}{2}$ term is the **Itô correction** (drift adjustment).  
Without it, the expected value would be wrong. This is one of the most important results in mathematical finance.

### Theoretical properties
$$E[S(t)] = S_0 e^{\mu t} \qquad \text{(mean grows exponentially at rate } \mu \text{)}$$
$$\text{Terminal prices } S(T) \sim \text{Log-Normal}$$

In [ ]:
# Simulate GBM: stock starting at 100, 8% return, 20% volatility, 1 year
gbm = GeometricBrownianMotion(S0=100, mu=0.08, sigma=0.20, T=1.0, N=252, n_paths=1000, seed=42)
S   = gbm.simulate()

stats = gbm.terminal_distribution(S)

print(f"All prices positive     : {np.all(S > 0)} ✅")
print()
print(f"Terminal mean  simulated: {stats['mean']:.2f}   (theory = {gbm.theoretical_mean()[-1]:.2f})")
print(f"Terminal median         : {stats['median']:.2f}")
print(f"Terminal std            : {stats['std']:.2f}")
print(f"5th / 95th percentile   : {stats['5th_pct']:.2f} / {stats['95th_pct']:.2f}")
print(f"P(S(T) > S0)            : {stats['prob_above_S0']:.1%}  (positive drift → majority end above S0)")

Key insight: the **mean > median** because the log-normal distribution is right-skewed.  
Most paths end near the median (~107), but a small number of very high outcomes pull the mean up to ~108.  
This is why the median is a better measure of a "typical" outcome for skewed financial returns.

In [ ]:
fig = plot_gbm(gbm, S)
plt.show()

**Reading the 4-panel figure:**
- **Top left** — 50 simulated price paths with the theoretical mean $E[S(t)] = S_0 e^{\mu t}$ in blue.
- **Top right** — Terminal price histogram vs the theoretical log-normal density — a key validation.
- **Bottom left** — Daily log-returns are normally distributed, confirming the Black-Scholes assumption.
- **Bottom right** — The percentile fan shows the range where 90% of paths end up. This is a **risk visualisation** used in real portfolio reports.

---
## 3. Ornstein-Uhlenbeck Process (Mean-Reverting)

GBM has no "memory" — it drifts wherever randomness takes it.  
But many financial variables are **mean-reverting** — they tend to return to a long-run average.

Examples:
- **Interest rates** — central banks push them back when they stray too far
- **Commodity prices** — high prices attract production; low prices reduce it
- **Pairs trading spreads** — two correlated assets drift apart then reconverge

The **Ornstein-Uhlenbeck process** models this mathematically.

### The SDE
$$dX = \theta(\mu - X)\,dt + \sigma\,dW$$

Where:
- $\theta$ — **speed of mean reversion** (how strongly X is pulled back to μ)
- $\mu$ — **long-run mean** (the equilibrium level)
- $\sigma$ — **volatility**

When $X > \mu$: the drift term $\theta(\mu - X)$ is **negative** → pulled down  
When $X < \mu$: the drift term $\theta(\mu - X)$ is **positive** → pulled up

### Exact discretisation (no approximation error)
Unlike Euler-Maruyama, the exact conditional distribution at each step is:
$$X(t+\Delta t) \mid X(t) \sim N\left(X(t)e^{-\theta\Delta t} + \mu(1-e^{-\theta\Delta t}),\; \sigma^2\frac{1-e^{-2\theta\Delta t}}{2\theta}\right)$$

### Theoretical properties
$$E[X(t)] = \mu + (X_0 - \mu)e^{-\theta t} \qquad \text{(converges to } \mu \text{ exponentially)}$$
$$\text{Var}[X(t)] = \frac{\sigma^2}{2\theta}(1 - e^{-2\theta t}) \qquad \text{(converges to long-run variance } \frac{\sigma^2}{2\theta}\text{)}$$

In [ ]:
# Simulate OU: interest rate starting at 8%, reverting to long-run mean of 5%
ou = OrnsteinUhlenbeck(X0=0.08, theta=3.0, mu=0.05, sigma=0.02, T=2.0, N=504, n_paths=1000, seed=42)
X  = ou.simulate()

long_run_var = ou.sigma**2 / (2 * ou.theta)

print(f"Starting value X0       : {ou.X0:.1%}")
print(f"Long-run mean μ         : {ou.mu:.1%}")
print()
print(f"Final mean  simulated   : {X[:, -1].mean():.4f}   (theory = {ou.theoretical_mean()[-1]:.4f})")
print(f"Final std   simulated   : {X[:, -1].std():.4f}   (theory = {np.sqrt(ou.theoretical_var()[-1]):.4f})")
print()
print(f"Long-run variance σ²/2θ : {long_run_var:.6f}")
print(f"Simulated terminal var  : {np.var(X[:, -1]):.6f}")

The process started at **8%** and has reverted to near **5%** by the end of the simulation.  
Simulated mean and variance match theory to 4 decimal places — confirming the exact discretisation works perfectly.

In [ ]:
fig = plot_ou(ou, X)
plt.show()

**Reading the plots:**
- **Left** — All paths are pulled toward μ = 5% over time. The ±1σ band narrows as the process stabilises.
- **Right** — The simulated mean (coral) tracks the theoretical mean (blue dashed) almost exactly across the entire time horizon — validating the exact discretisation.

---
## 4. Summary & Comparison

| Property | Brownian Motion | GBM | Ornstein-Uhlenbeck |
|---|---|---|---|
| Always positive? | ❌ | ✅ | ❌ |
| Mean reverting? | ❌ | ❌ | ✅ |
| Exact solution? | ✅ | ✅ | ✅ (this implementation) |
| Typical use | Foundation | Stock prices | Interest rates, spreads |
| Terminal distribution | Normal | Log-normal | Normal |
| Variance over time | Grows as t | Grows exponentially | Converges to σ²/2θ |

In [ ]:
# Side-by-side comparison: one path from each process (normalised for scale)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("One sample path from each process", fontsize=13, y=1.02)

titles = ["Brownian Motion\nW(0)=0, drifts freely",
          "Geometric Brownian Motion\nS(0)=100, always positive",
          "Ornstein-Uhlenbeck\nX(0)=8%, reverts to μ=5%"]
paths  = [W[0], S[0], X[0]]
ts     = [bm.t, gbm.t, ou.t]
colors = ["#185FA5", "#1D9E75", "#D85A30"]
hlines = [None, None, ou.mu]

for ax, title, path, t, color, hline in zip(axes, titles, paths, ts, colors, hlines):
    ax.plot(t, path, color=color, linewidth=1.2)
    if hline is not None:
        ax.axhline(hline, color="gray", linestyle="--", linewidth=1, alpha=0.7,
                   label=f"μ = {hline:.0%}")
        ax.legend(fontsize=9)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("Time (years)")
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

---
## Next Steps

This is **Module 01** of a 12-project quantitative finance portfolio.

- **Module 02**: Portfolio Optimiser — Markowitz efficient frontier, VaR, CVaR
- **Module 03**: Financial Forecaster — ARIMA, GARCH, LSTM

---
*Abubakar Mamudu Alutiba — [github.com/AbubakarMA](https://github.com/AbubakarMA)*